#### Ingesting Yellow Taxi Trip Data - April 2026

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [2]:
from pyspark.sql import SparkSession 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Yellow Taxi Pipeline")
    .config("spark.driver.memory", "4g")
    #.config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# Reduce verbose logging
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 19:13:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.2.0
App name      : Yellow Taxi Pipeline
Master        : local[*]


In [5]:
spark

#### Defining Schema for Yellow Taxi Trip Data

In [6]:
yellow_taxi_schema = '''
                        VendorID INTEGER,
                        tpep_pickup_datetime TIMESTAMP,
                        tpep_dropoff_datetime TIMESTAMP,
                        passenger_count LONG,
                        trip_distance DOUBLE,
                        RatecodeID LONG,
                        store_and_fwd_flag STRING,
                        PULocationID INTEGER,
                        DOLocationID INTEGER,
                        payment_type LONG,
                        fare_amount DOUBLE,
                        extra DOUBLE,
                        mta_tax DOUBLE,
                        tip_amount DOUBLE,
                        tolls_amount DOUBLE,
                        improvement_surcharge DOUBLE,
                        total_amount DOUBLE,
                        congestion_surcharge DOUBLE,
                        airport_fee DOUBLE,
                        cbd_congestion_fee DOUBLE
'''

#### Reading Yellow Taxi Trip Parquet file

In [7]:
yellow_taxi_data = spark.read.option('header',True)\
                                .schema(yellow_taxi_schema)\
                                    .parquet('/app/data/input/yellow/yellow_tripdata_2026-04.parquet')


In [8]:
yellow_taxi_data.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2026-04-01 00:40:05|  2026-04-01 00:52:44|              1|          2.8|         1|                 N|         237|    

In [9]:
yellow_taxi_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
yellow_taxi_data.rdd.getNumPartitions()

8

#### Data Quality Checks and Filtering

In [11]:
yellow_taxi_data.count()

3831240

##### Checking tip amount > 0 for cash payments --> not a valid case [drop them]

In [12]:

yellow_taxi_data = yellow_taxi_data.\
        filter(~((col('payment_type') == 2) & (col('tip_amount').cast("decimal") > 0.0)))

##### Filtering any row where pick up date is greater than 2026-04

In [14]:
yellow_taxi_data = yellow_taxi_data.where(~(date_format('tpep_pickup_datetime',format = 'yyyy-MM') >='2026-05'))

##### Filtering any row where pick up date is smaller than 2026-04

In [15]:
yellow_taxi_data = yellow_taxi_data.where(~(date_format('tpep_pickup_datetime',format = 'yyyy-MM') <'2026-04'))

##### Filtering rows where passenger count  == 0

In [16]:
yellow_taxi_data = yellow_taxi_data.filter(col('passenger_count') != 0)

##### Filtering rows for negative fare amount

In [17]:
yellow_taxi_data = yellow_taxi_data.filter(col('fare_amount') > 0.0)

##### checking if any trip with negative distance

In [18]:
yellow_taxi_data.filter(col('trip_distance') < 0.0).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

##### Checking if any trip with pickup time later than drop off time

In [19]:
yellow_taxi_data.where(col('tpep_pickup_datetime') > col('tpep_dropoff_datetime')).count()

0

In [20]:
yellow_taxi_data \
    .withColumn('pickup_date', date_format(col('tpep_pickup_datetime'), 'yyyy-MM-dd')) \
    .write \
    .option('header', True) \
    .partitionBy('pickup_date') \
    .mode('overwrite') \
    .parquet('/app/data/output/yellow')

In [ ]:
spark.stop()
del spark